In [ ]:
import importlib.util
import subprocess
import sys

def check_and_install(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    
    # Verificar si el paquete está instalado
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"✅ {package_name} instalado correctamente.")
    else:
        print(f"📦 {package_name} ya está presente.")

# Ejecutar comprobación e instalación
check_and_install("ultralytics")
check_and_install("opencv-python", "cv2")

# Mostrar versiones finales
import ultralytics
import cv2
print("-" * 30)
print(f"🚀 Versión de Ultralytics: {ultralytics.__version__}")
print(f"📷 Versión de OpenCV: {cv2.__version__}")
print("-" * 30)

# Verificación de hardware para YOLO
ultralytics.checks()

In [ ]:
from google.colab import drive
import os

# Conectar a Drive
drive.mount('/content/drive')

# Definir la ruta de la carpeta del proyecto
path_ppe = '/content/drive/MyDrive/ppe'

# Crear la carpeta 'ppe' si no existe
if not os.path.exists(path_ppe):
    os.makedirs(path_ppe)
    print(f"✅ Carpeta creada en: {path_ppe}")
else:
    print(f"📦 La carpeta '{path_ppe}' ya existe.")

In [ ]:
!pip install -q roboflow

In [ ]:
from roboflow import Roboflow

# Configuración de Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")

# Descargar el dataset en formato YOLOv8
# Nota: Cambia el número de versión (1, 2, etc.) según la que quieras bajar
dataset = project.version(1).download("yolov8")

# Ruta del archivo data.yaml para usar en el entrenamiento
print(f"Dataset descargado en: {dataset.location}")
path_yaml = f"{dataset.location}/data.yaml"

In [ ]:
!ls

In [ ]:
import os
import shutil
import yaml
from google.colab import drive
from roboflow import Roboflow
from ultralytics import YOLO

# 1. Montar Drive y preparar carpetas
drive.mount('/content/drive')
path_ppe = '/content/drive/MyDrive/ppe'
dataset_dest = os.path.join(path_ppe, "dataset_ppe")

if not os.path.exists(path_ppe):
    os.makedirs(path_ppe)

# 2. Descargar de Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
dataset = project.version(1).download("yolov8")

# 3. Mover dataset a Drive (si no existe ya)
if not os.path.exists(dataset_dest):
    shutil.move(dataset.location, dataset_dest)
    print(f"✅ Dataset movido a Drive: {dataset_dest}")
else:
    print("📦 El dataset ya existía en Drive.")

# 4. Corregir rutas en el archivo data.yaml (CRUCIAL para que YOLO no falle)
path_yaml = os.path.join(dataset_dest, "data.yaml")
with open(path_yaml, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = os.path.join(dataset_dest, "train/images")
config['val'] = os.path.join(dataset_dest, "valid/images")
if 'test' in config:
    config['test'] = os.path.join(dataset_dest, "test/images")

with open(path_yaml, 'w') as f:
    yaml.dump(config, f)



In [ ]:
# 5. Cargar Modelo YOLO11 e INICIAR ENTRENAMIENTO
model = YOLO("yolo8n.pt")

model.train(
    data=path_yaml,
    epochs=100,
    imgsz=640,
    project=path_ppe,        # Guarda resultados en Drive
    name="entrenamiento_ppe" # Nombre de la carpeta de resultados
)